# 🎙️ VoiceDiary AI — Live Cloud GPU Platform
### Bilingual Classroom Lecture Note-Taking & Speaker Diarization Engine
**VoiceDiary © 2026 Abdul Sarim Khan. All Rights Reserved.**

Powered by **NVIDIA Tesla T4 GPU Tensor Cores** (`large-v3-turbo` + `ECAPA-TDNN` + `Gemini 2.5 Flash`).

---
### ⚡ Quick Start:
1. In the menu bar above, click **Runtime → Run all** (or press `Ctrl + F9`).
2. Click the public **Gradio Live URL** generated at the bottom to open the full-fidelity web application!

In [ ]:
# 1. Install GPU Acceleration Libraries
!pip install -q faster-whisper speechbrain gradio torchaudio soundfile scipy


In [ ]:
# 2. Launch Full-Fidelity VoiceDiary Web Platform (Exact Desktop UI/UX)
import os, time, tempfile, json, urllib.request
import numpy as np
import soundfile as sf
import gradio as gr
import torch
from faster_whisper import WhisperModel
from speechbrain.inference.speaker import EncoderClassifier

# Hardware Acceleration Telemetry
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU Multi-Core (AVX2)'
compute_dtype = 'float16' if torch.cuda.is_available() else 'int8'
device_type = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'⚡ Engine Online: {gpu_name} ({compute_dtype})')

# Global Model Cache
loaded_models = {}

def get_whisper_model(model_name='large-v3-turbo'):
    if model_name not in loaded_models:
        print(f'Loading Whisper {model_name} on {device_type}...')
        loaded_models[model_name] = WhisperModel(model_name, device=device_type, compute_type=compute_dtype)
    return loaded_models[model_name]

# Pre-load Large-v3-Turbo
whisper_model = get_whisper_model('large-v3-turbo')

# Load ECAPA-TDNN Diarizer on GPU
with tempfile.TemporaryDirectory() as tmp:
    embedder = EncoderClassifier.from_hparams(
        source='speechbrain/spkrec-ecapa-voxceleb',
        savedir=tmp,
        run_opts={'device': device_type}
    )

# Full Obsidian & Amber Gold Desktop CSS
custom_css = """
@import url('https://fonts.googleapis.com/css2?family=Plus+Jakarta+Sans:wght@400;500;600;700;800&family=Fira+Code:wght@500;600&family=Noto+Nastaliq+Urdu:wght@400;700&display=swap');

:root {
    --bg-primary: #090D16 !important;
    --bg-card: #0F172A !important;
    --border-color: #1E293B !important;
    --accent-gold: #F59E0B !important;
}

body, .gradio-container {
    background-color: #090D16 !important;
    font-family: 'Plus Jakarta Sans', sans-serif !important;
    color: #F8FAFC !important;
    max-width: 1400px !important;
    margin: 0 auto !important;
}

/* Header */
.vd-header {
    background: #0F172A;
    border: 1px solid #1E293B;
    border-radius: 16px;
    padding: 16px 24px;
    margin-bottom: 18px;
    display: flex;
    justify-content: space-between;
    align-items: center;
    box-shadow: 0 10px 30px -10px rgba(0,0,0,0.6);
}

.brand-wrapper {
    display: flex;
    align-items: center;
    gap: 14px;
}

.brand-logo-badge {
    width: 44px;
    height: 44px;
    background: linear-gradient(135deg, #F59E0B, #D97706);
    border-radius: 12px;
    display: flex;
    align-items: center;
    justify-content: center;
    font-size: 22px;
    box-shadow: 0 0 20px rgba(245, 158, 11, 0.35);
}

.brand-title {
    font-size: 20px;
    font-weight: 800;
    color: #FFFFFF;
    letter-spacing: -0.02em;
    margin: 0;
}

.brand-sub {
    font-size: 11px;
    color: #94A3B8;
    font-weight: 600;
}

.hw-badge-pill {
    display: inline-flex;
    align-items: center;
    gap: 8px;
    padding: 6px 16px;
    border-radius: 9999px;
    background: rgba(16, 185, 129, 0.1);
    border: 1px solid rgba(16, 185, 129, 0.3);
    color: #34D399;
    font-size: 12px;
    font-weight: 700;
    font-family: 'Fira Code', monospace;
}

/* Panels */
.vd-card {
    background: #0F172A !important;
    border: 1px solid #1E293B !important;
    border-radius: 16px !important;
    padding: 20px !important;
}

.btn-gold-primary {
    background: linear-gradient(135deg, #F59E0B, #D97706) !important;
    color: #000000 !important;
    font-weight: 800 !important;
    font-size: 15px !important;
    border: none !important;
    border-radius: 12px !important;
    padding: 14px 24px !important;
    box-shadow: 0 0 20px rgba(245, 158, 11, 0.3) !important;
    cursor: pointer !important;
    transition: all 0.2s ease !important;
}
.btn-gold-primary:hover {
    transform: translateY(-2px) !important;
    box-shadow: 0 0 30px rgba(245, 158, 11, 0.5) !important;
}

.speaker-item {
    background: rgba(255, 255, 255, 0.03);
    border: 1px solid rgba(255, 255, 255, 0.06);
    border-radius: 12px;
    padding: 12px 14px;
    display: flex;
    align-items: center;
    gap: 12px;
    margin-bottom: 10px;
}

.spk-avatar {
    width: 36px;
    height: 36px;
    border-radius: 50%;
    display: flex;
    align-items: center;
    justify-content: center;
    font-weight: 800;
    font-size: 14px;
    color: #FFFFFF;
}

.transcript-panel {
    background: #0F172A;
    border: 1px solid #1E293B;
    border-radius: 16px;
    padding: 24px;
    min-height: 520px;
    max-height: 650px;
    overflow-y: auto;
}
"""

def run_pipeline(audio_path, model_choice, lang_choice, sim_threshold):
    if not audio_path or not os.path.exists(audio_path):
        return "<div style='color:#EF4444;padding:30px;text-align:center;'>⚠️ Please record audio or upload a classroom lecture file.</div>", "<div style='color:#64748B;padding:20px;text-align:center;'>No speakers active</div>", None, None, None
    
    t0 = time.time()
    
    # Load Model Dynamically
    model_key_map = {
        '⚡ Large-v3-Turbo (809M) - OpenAI SOTA': 'large-v3-turbo',
        '🚀 Whisper Base (74M) - Fast': 'base',
        '⚡ Whisper Tiny (39M) - Ultralight': 'tiny',
        '🎯 Whisper Small (244M) - High Accuracy': 'small',
        '🔬 Whisper Medium (769M) - Deep Precision': 'medium',
        '⚡ Distil-Whisper (756M) - English Fast': 'distil-large-v3'
    }
    selected_model_key = model_key_map.get(model_choice, 'large-v3-turbo')
    model = get_whisper_model(selected_model_key)
    
    # Read and resample audio
    data, sr = sf.read(audio_path)
    if data.ndim > 1:
        data = data.mean(axis=1)
    data = data.astype(np.float32)
    
    if sr != 16000:
        from scipy.signal import resample_poly
        gcd = int(np.gcd(16000, sr))
        data = resample_poly(data, 16000 // gcd, sr // gcd).astype(np.float32)
        
    duration = len(data) / 16000.0
    
    lang_map = {
        '🌐 Bilingual (Auto Urdu + English)': None,
        '🇵🇰 Pure Urdu Script (اردو)': 'ur',
        '🇬🇧 English Only': 'en'
    }
    target_lang = lang_map.get(lang_choice, None)
    
    # Transcribe with Whisper
    segments, info = model.transcribe(
        data,
        beam_size=3,
        language=target_lang,
        vad_filter=True,
        vad_parameters=dict(min_silence_duration_ms=400),
    )
    
    speaker_profiles = {}
    speaker_names = {}
    speaker_colors = ['#818CF8', '#34D399', '#FBBF24', '#F87171', '#C084FC', '#38BDF8']
    next_speaker_id = 1
    
    html_bubbles = []
    txt_lines = [f'VoiceDiary Lecture Notes ({duration:.1f}s)\n']
    md_lines = [f'# VoiceDiary Lecture Notes\n*Duration: {duration:.1f}s | Model: {selected_model_key}*\n']
    
    threshold = float(sim_threshold) / 100.0 if sim_threshold else 0.32
    
    for seg in segments:
        text = seg.text.strip()
        if not text: continue
        start_s, end_s = seg.start, seg.end
        seg_audio = data[int(start_s*16000):int(end_s*16000)]
        
        spk_id = 1
        if len(seg_audio) >= 8000:
            with torch.inference_mode():
                w = torch.from_numpy(seg_audio).float().unsqueeze(0).to(embedder.device)
                emb = embedder.encode_batch(w).squeeze().detach().cpu().numpy()
                emb_norm = emb / (np.linalg.norm(emb) + 1e-9)
            
            best_id, best_sim = None, -1.0
            for s_id, embs in speaker_profiles.items():
                sims = [float(np.dot(emb_norm, e)) for e in embs]
                m = max(sims) if sims else 0
                if m > best_sim:
                    best_sim, best_id = m, s_id
            
            if best_id and best_sim >= threshold:
                spk_id = best_id
                if len(speaker_profiles[spk_id]) < 50:
                    speaker_profiles[spk_id].append(emb_norm)
            else:
                spk_id = next_speaker_id
                speaker_profiles[spk_id] = [emb_norm]
                speaker_names[spk_id] = f'Speaker {spk_id}'
                next_speaker_id += 1
                
        color = speaker_colors[(spk_id - 1) % len(speaker_colors)]
        spk_name = speaker_names.get(spk_id, f'Speaker {spk_id}')
        time_str = f'{int(start_s//60):02d}:{int(start_s%60):02d}'
        
        bubble = f"""<div style="padding:16px 20px; border-radius:12px; background:rgba(255,255,255,0.02); border:1px solid rgba(255,255,255,0.06); border-left:4px solid {color}; margin-bottom:14px;">
            <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:6px;">
                <span style="color:{color}; font-weight:800; font-size:13px;">● {spk_name}</span>
                <span style="color:#64748B; font-family:'Fira Code',monospace; font-size:12px;">{time_str}</span>
            </div>
            <div style="font-size:14px; line-height:1.6; color:#E2E8F0;">{text}</div>
        </div>"""
        html_bubbles.append(bubble)
        txt_lines.append(f'[{time_str}] {spk_name}: {text}')
        md_lines.append(f'**{spk_name}** `[{time_str}]`: {text}\n')
        
    elapsed = time.time() - t0
    speedup = duration / max(0.01, elapsed)
    
    perf_banner = f"""<div style="padding:14px 18px; border-radius:12px; background:rgba(245,158,11,0.08); border:1px solid rgba(245,158,11,0.25); color:#FBBF24; font-size:13px; font-weight:700; margin-top:20px; display:flex; justify-content:space-between; font-family:'Fira Code',monospace;">
        <span>⚡ INFERENCE: {elapsed:.2f}s</span>
        <span>🚀 SPEEDUP: {speedup:.1f}x REAL-TIME</span>
        <span>🎯 MODEL: {selected_model_key.upper()}</span>
    </div>"""
    html_bubbles.append(perf_banner)
    
    # Generate Speaker Profiles Sidebar HTML
    spk_cards = []
    for s_id, embs in speaker_profiles.items():
        c = speaker_colors[(s_id - 1) % len(speaker_colors)]
        spk_cards.append(f"""<div class="speaker-item">
            <div class="spk-avatar" style="background:{c};">{s_id}</div>
            <div>
                <div style="font-weight:700; font-size:13px; color:#FFFFFF;">Speaker {s_id}</div>
                <div style="font-size:11px; color:#94A3B8;">{len(embs)} voice prints enrolled</div>
            </div>
        </div>""")
    sidebar_html = "\n".join(spk_cards) if spk_cards else "<div style='color:#64748B;padding:20px;text-align:center;'>No speakers enrolled</div>"
    
    full_html = "\n".join(html_bubbles)
    full_txt = "\n".join(txt_lines)
    full_md = "\n".join(md_lines)
    
    f_md = tempfile.NamedTemporaryFile(mode='w', suffix='.md', delete=False, encoding='utf-8')
    f_md.write(full_md); f_md.close()
    
    f_txt = tempfile.NamedTemporaryFile(mode='w', suffix='.txt', delete=False, encoding='utf-8')
    f_txt.write(full_txt); f_txt.close()
    
    return full_html, sidebar_html, f_md.name, f_txt.name, full_md

# Gemini AI Summarizer Call
def generate_gemini_summary(transcript_text, gemini_key):
    key = (gemini_key or os.environ.get('GEMINI_API_KEY', '')).strip()
    if not key:
        return "❌ Please provide a Gemini API Key."
    if not transcript_text or len(transcript_text) < 20:
        return "⚠️ No transcript available to summarize. Transcribe a lecture first."
        
    url = f'https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?key={key}'
    prompt = f"""You are an academic study assistant for VoiceDiary. Analyze this university classroom lecture transcript and format structured study notes with:
1. 📌 Executive Lecture Summary (3-4 bullet points)
2. 🎯 Key Concepts & Formulas Discussed
3. 💡 Exam Study Flashcards (5 Q&A pairs)
4. 📋 Homework & Assignments Mentioned

Transcript:
{transcript_text}
"""
    payload = {'contents': [{'parts': [{'text': prompt}]}]}
    req = urllib.request.Request(url, data=json.dumps(payload).encode('utf-8'), headers={'Content-Type': 'application/json'})
    
    try:
        with urllib.request.urlopen(req, timeout=25) as resp:
            data = json.loads(resp.read().decode('utf-8'))
            return data['candidates'][0]['content']['parts'][0]['text']
    except Exception as e:
        return f"❌ Gemini Error: {e}"

# Build Full Web GUI
with gr.Blocks(title="VoiceDiary — AI Classroom Diarization", css=custom_css) as demo:
    transcript_state = gr.State("")
    
    gr.HTML(f"""
    <div class="vd-header">
        <div class="brand-wrapper">
            <div class="brand-logo-badge">🎙️</div>
            <div>
                <div class="brand-title">VoiceDiary</div>
                <div class="brand-sub">AI Bilingual Lecture & Diarization Engine © Abdul Sarim Khan</div>
            </div>
        </div>
        <div style="display:flex; align-items:center; gap:16px;">
            <div class="hw-badge-pill">⚡ {gpu_name} (Tensor Cores FP16)</div>
        </div>
    </div>
    """)
    
    with gr.Row():
        # LEFT COLUMN: Speakers & Controls (Matches Desktop Sidebar)
        with gr.Column(scale=3):
            with gr.Group(elem_classes=["vd-card"]):
                gr.HTML("<div style='font-size:13px; font-weight:800; color:#F59E0B; margin-bottom:12px; letter-spacing:0.05em;'>SPEAKERS & PROFILES</div>")
                sidebar_out = gr.HTML(value="<div style='color:#64748B;padding:16px;text-align:center;'>Start transcription to enroll speaker voiceprints</div>")
                
            with gr.Group(elem_classes=["vd-card"], visible=True):
                gr.HTML("<div style='font-size:13px; font-weight:800; color:#F59E0B; margin-bottom:12px; letter-spacing:0.05em;'>AI MODEL & ENGINE SETTINGS</div>")
                model_dropdown = gr.Dropdown(
                    choices=[
                        '⚡ Large-v3-Turbo (809M) - OpenAI SOTA',
                        '🚀 Whisper Base (74M) - Fast',
                        '⚡ Whisper Tiny (39M) - Ultralight',
                        '🎯 Whisper Small (244M) - High Accuracy',
                        '🔬 Whisper Medium (769M) - Deep Precision',
                        '⚡ Distil-Whisper (756M) - English Fast'
                    ],
                    value='⚡ Large-v3-Turbo (809M) - OpenAI SOTA',
                    label='🧠 Active Whisper Model'
                )
                lang_dropdown = gr.Dropdown(
                    choices=[
                        '🌐 Bilingual (Auto Urdu + English)',
                        '🇵🇰 Pure Urdu Script (اردو)',
                        '🇬🇧 English Only'
                    ],
                    value='🌐 Bilingual (Auto Urdu + English)',
                    label='🗣️ Language Mode'
                )
                thresh_slider = gr.Slider(minimum=20, maximum=70, value=32, step=1, label='🎯 Diarization Sensitivity (Cosine Threshold %)')
                gemini_key_in = gr.Textbox(placeholder='Paste Gemini API Key (Optional)...', type='password', label='✨ Gemini AI Key (BYOK)')

        # RIGHT COLUMN: Main Audio Input & Real-Time Classroom Transcript
        with gr.Column(scale=7):
            with gr.Group(elem_classes=["vd-card"]):
                with gr.Tabs():
                    with gr.TabItem("🎙️ Live Classroom Microphone"):
                        audio_mic = gr.Audio(sources=["microphone"], type="filepath", label="Capture Classroom Speech")
                    with gr.TabItem("📁 Upload Lecture Audio File"):
                        audio_file = gr.Audio(sources=["upload"], type="filepath", label="Upload Audio File (.wav, .mp3, .m4a, .flac)")
                
                transcribe_btn = gr.Button("🚀 Transcribe & Diarize (GPU Accelerated)", elem_classes=["btn-gold-primary"])
            
            with gr.Group(elem_classes=["vd-card"]):
                gr.HTML("<div style='font-size:14px; font-weight:700; color:#FFFFFF; margin-bottom:10px;'>📝 Live Diarized Classroom Lecture Notes</div>")
                transcript_display = gr.HTML(
                    value="<div style='display:flex; flex-direction:column; align-items:center; justify-content:center; height:320px; color:#64748B; text-align:center;'><div style='font-size:40px; margin-bottom:12px;'>🎙️</div><div>Lecture transcript will stream here.<br/>Click the gold button above to begin.</div></div>",
                    elem_classes=["transcript-panel"]
                )
            
            # Export & Gemini AI Summary
            with gr.Group(elem_classes=["vd-card"]):
                gr.HTML("<div style='font-size:13px; font-weight:800; color:#F59E0B; margin-bottom:10px;'>📥 EXPORT & AI LECTURE STUDY NOTES</div>")
                with gr.Row():
                    d_md = gr.File(label="📝 Markdown Notes (.md)")
                    d_txt = gr.File(label="📄 Plain Text (.txt)")
                
                ai_sum_btn = gr.Button("✨ Generate AI Lecture Summary & Flashcards (Gemini 2.5 Flash)")
                summary_out = gr.Markdown(value="*AI summary and study flashcards will appear here after clicking above...*")

    # Wire event handlers
    transcribe_btn.click(
        fn=lambda m, f, mod, lang, th: run_pipeline(m if m else f, mod, lang, th),
        inputs=[audio_mic, audio_file, model_dropdown, lang_dropdown, thresh_slider],
        outputs=[transcript_display, sidebar_out, d_md, d_txt, transcript_state]
    )
    
    ai_sum_btn.click(
        fn=generate_gemini_summary,
        inputs=[transcript_state, gemini_key_in],
        outputs=[summary_out]
    )

demo.launch(share=True)
